# P2.1 · MLP binary classifier on two moons

**ID:** P2.1 · **Target:** 25 min · **Plan week:** 1 (and timed repeats) · **Part:** P (PyTorch first principles)

**The drill.** From a blank notebook, build the full P0 skeleton for a 2-D binary
classification problem: data → split → normalize → `DataLoader` → `nn.Module` →
`BCEWithLogitsLoss` + AdamW → train/eval loops → sanity checks → decision-boundary plot.
Talk out loud as if an interviewer were watching.

**Must include (from the plan):** `BCEWithLogitsLoss`, accuracy, decision-boundary plot.
Also: initial-loss check (≈ ln 2 ≈ 0.693), overfit-one-batch check, `model.eval()` + `torch.no_grad()` for eval.

**Rules.** No docs, no autocomplete-from-LLM. Each cell below has its spec as comments; delete
`# YOUR CODE HERE` and write it. The **self-check** cell at the end defines the variable names
it expects; it is the only "grader". Then answer the questions out loud and log the session.

## Sources
- PyTorch tutorial, *Learn the Basics* (Datasets & DataLoaders, Build the Neural Network,
  Optimizing Model Parameters): https://docs.pytorch.org/tutorials/beginner/basics/intro.html
  (BSD-3; the skeleton order follows its train/test loop structure).
- A. Karpathy, *A Recipe for Training Neural Networks* (2019), §2 "verify loss @ init",
  "overfit one batch", "add significant digits to your eval": https://karpathy.github.io/2019/04/25/recipe/
- `torch.nn.BCEWithLogitsLoss` docs (log-sum-exp stable form, `pos_weight`):
  https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html
- scikit-learn `make_moons` and `train_test_split(stratify=...)`:
  https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html
- Decision-boundary plotting via a mesh grid, as in the scikit-learn classifier comparison example:
  https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from prep import Session, check, seed_everything

s = Session("P2.1", target_min=25)
seed_everything(0)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 1. DATA. sklearn.datasets.make_moons(n_samples=2000, noise=0.25, random_state=0).
#    Inspect shapes, dtypes, class balance. Stratified split 70/15/15 -> X_train, X_val, X_test (+ y_*).
#    Standardize with TRAIN mean/std only (store them as `mu`, `sd`). Convert to float32 tensors.
# YOUR CODE HERE


In [ ]:
# 2. DATALOADERS. TensorDataset + DataLoader; shuffle train only; batch size 64.
# YOUR CODE HERE


In [ ]:
# 3. MODEL. nn.Module subclass `MLP`: 2 -> 64 -> 64 -> 1, ReLU. Output RAW LOGITS of shape (B,).
#    Instantiate as `model`, move to `dev`, print the parameter count.
# YOUR CODE HERE


In [ ]:
# 4. LOSS + OPTIMIZER. BCEWithLogitsLoss (no sigmoid in the model!). AdamW lr=3e-3, weight_decay=1e-2.
# YOUR CODE HERE


In [ ]:
# 5. EVAL FUNCTION. evaluate(model, dl) -> (mean loss, accuracy) over the WHOLE set:
#    model.eval(), torch.no_grad(), sum losses weighted by batch size, count correct (logit > 0).
# YOUR CODE HERE


In [ ]:
# 6. SANITY CHECKS before training.
#    (a) initial val loss should be ~ln 2. Store it in `init_loss`.
#    (b) overfit ONE batch of 32 with a fresh model for 300 steps -> loss < 0.05. Store final as `overfit_loss`.
# YOUR CODE HERE


In [ ]:
# 7. TRAIN LOOP. 60 epochs. Each step: model.train(), to(dev), forward, loss, zero_grad, backward, step.
#    Record per-epoch mean train loss and val (loss, acc) in a dict `history`
#    with keys "train_loss", "val_loss", "val_acc". Keep the best-val-acc state_dict (early-stopping style).
# YOUR CODE HERE


In [ ]:
# 8. TEST + PLOTS. test_loss, test_acc = evaluate(model, test_dl) (touch test ONCE).
#    Plot (left) train/val loss curves; (right) decision boundary: predict sigmoid(logit) on a
#    300x300 mesh over the standardized input range, contourf at levels [0, .5, 1], scatter test points.
# YOUR CODE HERE


## Self-check
Expects: `mu`, `sd`, `X_train`, `model`, `init_loss`, `overfit_loss`, `history`, `test_acc`.

In [ ]:
check("normalized with train stats", torch.allclose(X_train.mean(0), torch.zeros(2), atol=1e-4), "train mean should be ~0")
check("model has no Sigmoid layer", not any(isinstance(m, nn.Sigmoid) for m in model.modules()), "return logits; BCEWithLogits applies sigmoid")
out = model(X_train[:5].to(dev))
check("logits shape is (B,)", out.shape == (5,), f"got {tuple(out.shape)}; (B,1) vs (B,) silently broadcasts in BCE")
check("initial loss ~ ln 2", abs(init_loss - math.log(2)) < 0.15, f"got {init_loss:.3f}")
check("can overfit one batch", overfit_loss < 0.05, f"got {overfit_loss:.4f}")
check("history recorded", len(history["train_loss"]) > 0 and len(history["val_acc"]) == len(history["train_loss"]))
check("test accuracy > 0.90", test_acc > 0.90, f"got {test_acc:.3f}")

## Say it out loud (P3)
1. Why does the model output logits and use `BCEWithLogitsLoss` instead of `Sigmoid` + `BCELoss`?
2. Why is the expected initial loss ln 2 here? What would you expect for C classes?
3. What does `model.eval()` change for *this* model? What does `torch.no_grad()` save?
4. Why compute val accuracy as total-correct / total-N rather than averaging per-batch accuracies?
5. Why standardize with train statistics only? What would leak otherwise?
6. What silently broadcasts if the model returns shape `(B, 1)` and targets are `(B,)`?

In [ ]:
s.finish(lookups=0, score=None, notes="")  # score: 1-5 self-rating; notes: what tripped you up